# 00 Quickstart

## Cosa fa / Cosa NON fa

- esegue la pipeline solo se abiliti esplicitamente `RUN_TOOLKIT = True`
- cerca un mart leggibile e mostra schema, anteprima e controlli minimi
- non scrive file e non assume output già presenti in una cartella specifica

In [ ]:
from pathlib import Path
import subprocess
import duckdb

ROOT = Path('.').resolve()
DATASET_YML = (ROOT / '..' / 'dataset.yml').resolve()
MART_DIRS = [
    (ROOT / '..' / 'data' / 'mart').resolve(),
    (ROOT / '..' / '_runs').resolve(),
]
TABLE_NAME = 'project_summary'
RUN_TOOLKIT = False
ROOT, DATASET_YML

In [ ]:
cmd = ['toolkit', 'run', '--dataset', str(DATASET_YML)]
if RUN_TOOLKIT:
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Toolkit run disabled.')
    print('Set RUN_TOOLKIT = True to execute the pipeline from this notebook.')
    print('Expected command:', ' '.join(cmd))

In [ ]:
def find_mart_candidates(table_name):
    patterns = [
        f'**/{table_name}.parquet',
        f'**/*{table_name}*.parquet',
    ]
    found = []
    for base in MART_DIRS:
        if not base.exists():
            continue
        for pattern in patterns:
            found.extend(sorted(base.glob(pattern)))
    unique = []
    seen = set()
    for path in found:
        key = str(path)
        if key not in seen:
            seen.add(key)
            unique.append(path)
    return unique

mart_files = find_mart_candidates(TABLE_NAME)
mart_files[:5]

In [ ]:
con = duckdb.connect()
mart_path = str(mart_files[0]) if mart_files else None
if mart_path:
    schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{mart_path}')").df()
    head_df = con.execute(f"SELECT * FROM read_parquet('{mart_path}') LIMIT 10").df()
    display(schema_df)
    display(head_df)
else:
    print('No mart parquet found. Run the pipeline first or update TABLE_NAME.')

In [ ]:
def read_schema(path):
    schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()
    schema.columns = [str(col).lower() for col in schema.columns]
    return schema

def choose_columns(schema):
    name_col = 'column_name' if 'column_name' in schema.columns else schema.columns[0]
    type_col = 'column_type' if 'column_type' in schema.columns else schema.columns[1]
    rows = [
        {'name': str(row[name_col]), 'type': str(row[type_col]).upper()}
        for _, row in schema.iterrows()
    ]
    year_col = next((r['name'] for r in rows if r['name'].lower() == 'year' or 'anno' in r['name'].lower()), None)
    numeric_rows = [r for r in rows if any(token in r['type'] for token in ['INT', 'DECIMAL', 'DOUBLE', 'FLOAT', 'REAL', 'BIGINT'])]
    metric_col = next((r['name'] for r in numeric_rows if any(token in r['name'].lower() for token in ['value', 'tot', 'importo', 'ammontare', 'pct', 'percent'])), None)
    if metric_col is None and numeric_rows:
        metric_col = numeric_rows[0]['name']
    return year_col, metric_col

if mart_path:
    schema_df = read_schema(mart_path)
    YEAR_COL, METRIC_COL = choose_columns(schema_df)
    print({'YEAR_COL': YEAR_COL, 'METRIC_COL': METRIC_COL})

In [ ]:
if mart_path:
    row_count = con.execute(f"SELECT COUNT(*) AS row_count FROM read_parquet('{mart_path}')").df()
    display(row_count)

    if YEAR_COL:
        distinct_count = con.execute(
            f"SELECT COUNT(DISTINCT {YEAR_COL}) AS distinct_key_count FROM read_parquet('{mart_path}')"
        ).df()
        display(distinct_count)
    else:
        print('No year-like key detected. Update the helper or inspect schema_df manually.')

    check_columns = [col for col in [YEAR_COL, METRIC_COL] if col]
    if check_columns:
        null_expr = ', '.join([f"AVG(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS {col}_null_rate" for col in check_columns])
        null_rates = con.execute(f"SELECT {null_expr} FROM read_parquet('{mart_path}')").df()
        display(null_rates)
    else:
        print('No suitable columns found for null-rate sanity checks.')